In [52]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [53]:
import os
import json 
import hashlib
import chromadb

from typing import List, Dict,Any, Optional, Tuple
from dataclasses import dataclass, field
from datetime import datetime

from chromadb.utils import embedding_functions
from chromadb.api.models.Collection import Collection
from dotenv import load_dotenv
from tavily import TavilyClient
from pydantic import BaseModel,Field

In [54]:
from tavily import TavilyClient
import os

tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

In [55]:

load_dotenv()
assert os.getenv('OPENAI_API_KEY') is not None
assert os.getenv('TAVILY_API_KEY') is not None

In [56]:
def search_games(query,top_k=2):
    return collection.query(
        query_texts=[query],
        n_results=top_k
    )

In [57]:
# Retrieve from Vector DB 

def retreive_game(query):
    return search_games(query)



In [58]:
# Evaluate retreival 

def evaluate_retrieval(results, threshold=0.5):
    try:
        if not results or "distances" not in results:
            return False, 0.0

        distance = results["distances"][0][0]
        score = 1 - distance
        return score >= threshold, score

    except Exception:
        return False, 0.0


In [59]:
# Web Search Fallback 

def game_web_search(query):
    return tavily.search(query=query, max_results=3)

In [60]:
def extract_game(results):
    try:
        if results and results.get("metadatas"):
            return results["metadatas"][0][0]
        return None

    except Exception:
        return None




In [61]:
def format_answer(game):
    return(
        f"{game.get('name','Unknown')} is a {game.get('genre','N/A')} game"
        f"released in {game.get('release_date','N/A')} for {game.get('platform','N/A')}."
        f"It was published by {game.get('publisher','N/A')}."
        f"{game.get('description', '')}"
    )


In [62]:
def format_web(res):
    try:
        if not res or not res.get("results"):
            return "No information found", "N/A"

        top=res["results"][0]
        return top.get("content","No content"),top.get("url","N/A")

    except Exception:
        return "No information found", "N/A"
    


In [63]:
from enum import Enum


In [64]:
class AgentState(Enum):
  START="START"
  RETRIEVE="RETRIEVE"
  EVALUATE="EVALUATE"
  WEB_SEARCH="WEB_SEARCH"
  FINAL="FINAL"


In [68]:
class UdaPlayAgent:
    def __init__(self):
        self.memory = []
        self.state_trace = []

    def transition(self, state):
        self.state_trace.append(state.value)

    def ask(self, query):
        self.state_trace = []
        self.transition(AgentState.START)

        self.transition(AgentState.RETRIEVE)
        results = retreive_game(query)

        self.transition(AgentState.EVALUATE)
        game = extract_game(results)
        ok, confidence = evaluate_retrieval(results)

        if ok and game:
            answer = format_answer(game)
            source = "Vector DB"
            citation = {
                "name": game.get("name"),
                "platform": game.get("platform")
            }
        else:
            # Web Search Fallback
            self.transition(AgentState.WEB_SEARCH)
            res = game_web_search(query)
            answer, url = format_web(res)

            source = "Web"
            citation = {"url": url}

        self.transition(AgentState.FINAL)
        output={
            "query": query,
            "answer": answer,
            "confidence": round(confidence,2),
            "source": source,
            "citation": citation,
            "state_trace": self.state_trace
       }

        self.memory.append(output)
        return output


In [69]:
 def generate_report(output):
    return f'''

    Query:
    {output['query']}

    Answer:
    {output['answer']}

    Confidence Score:
    {output['confidence']}

    Source:
    {output['source']}

    Citation
    {output['citation']}

    Execution Path :
    {"->".join(output['state_trace'])}

    '''

     

In [71]:
###### Test 

agent= UdaPlayAgent()

result=agent.ask("Tell me about Elden Ring")

print(generate_report(result))



   Query:
   Tell me about Elden Ring

   Answer:
   The Elden Ring (エルデンリング, Eruden Ringu) is a mysterious artifact that defines the world of Elden Ring itself.[1] It is the source of the Erdtree,[2]

   Confidence Score:
   0.0

   Source:
   Web

   Citation
   {'url': 'https://eldenring.fandom.com/wiki/Elden_Ring_(concept)'}

   Execution Path :
   START->RETRIEVE->EVALUATE->WEB_SEARCH->FINAL

   


In [72]:
###### Test 

agent= UdaPlayAgent()

result=agent.ask("Tell me about Pokémon Ruby and Sapphire")

print(generate_report(result))



   Query:
   Tell me about Pokémon Ruby and Sapphire

   Answer:
   This is my first time diving into Pokémon Ruby and Sapphire. I learned a lot. I know it wasn't all gameplay, but I like to tell a little bit

   Confidence Score:
   0.0

   Source:
   Web

   Citation
   {'url': 'https://www.youtube.com/watch?v=e_t-iZISt7k'}

   Execution Path :
   START->RETRIEVE->EVALUATE->WEB_SEARCH->FINAL

   


In [70]:
###### Test 

agent= UdaPlayAgent()

result=agent.ask("Tell me about Super Mario 64")

print(generate_report(result))



   Query:
   Tell me about Super Mario 64

   Answer:
   Super Mario 64 is a 3D action-adventure platform game released for the Nintendo 64 in 1996 for Japan and North America and in 1997 for Europe and Australia.

   Confidence Score:
   0.0

   Source:
   Web

   Citation
   {'url': 'https://www.mariowiki.com/Super_Mario_64'}

   Execution Path :
   START->RETRIEVE->EVALUATE->WEB_SEARCH->FINAL

   
